In [9]:
import sqlite3
import sys
import pymysql
pymysql.install_as_MySQLdb()
import os
import re
from os import path


tables = ['DetectorGeo','PmtBlr','ChannelGain','ChannelMapping','ChannelMask',
          'PmtNoiseRms','ChannelPosition',
          'PMTFEMapping', 'PMTFELowFrequencyNoise']


def create_table_sqlite(cursorSqlite, connSqlite, table_path, table):
    #cursorMySql.execute('show create table {}'.format(table))
    with open(table_path, 'r') as infile:
        lines = infile.read().split(";")
        for line in lines:
            sql  = line
            
            # show create table will include comments for each row
            # Example: `MinRun` int(11) NOT NULL COMMENT 'Minimum run number for which valid'
            # that is not compatible with sqlite3, so we remove it.
            sql = re.sub(r" COMMENT\s+\'.*\'", "", sql)

            # it will also include: ENGINE=MyISAM DEFAULT CHARSET=latin1
            # this is not compatible either, so we remove it.
            sql = re.sub(r"\s*ENGINE.*", "", sql)

            # some tables may have KEYs defined: KEY `ElecID` (`SensorID`)
            # this syntax is different, so we remove it too.
            # This happens for instance in table ChannelGain
            sql = re.sub(r",\s*\n\s*KEY.*[\n,]", "", sql)
            #print(sql)
            cursorSqlite.execute(sql)
            connSqlite.commit()

def copy_all_rows(connSqlite, cursorSqlite, cursorMySql, table):
    # Get all data
    cursorMySql.execute('SELECT * from {0}'.format(table))
    data = cursorMySql.fetchall()

    # Insert all rows
    fields = '?'
    try:
        nfields = len(data[0])
        fields += (nfields-1) * ',?'
        cursorSqlite.executemany('INSERT INTO {0} VALUES({1})'.format(table,fields),data)
        connSqlite.commit()
    except IndexError:
        print('Table ' +table+' is empty.')


def loadDB(dbname : str, fRemove=False):
    print("Cloning database {}".format(dbname))
    dbfile = path.join('localdb.'+dbname+'.sqlite3')
    
    if fRemove:
        try:
            os.remove(dbfile)
        except:
            pass

    connSqlite = sqlite3.connect(dbfile)
    cursorSqlite = connSqlite.cursor()

    for table in tables:
        print('{}/{}'.format(dbname, table)+'.sql')
        table_path = '{}/{}'.format(dbname, table)+'.sql'
        # Create table
        create_table_sqlite(cursorSqlite, connSqlite, table_path, table)

if __name__ == '__main__':
    dbname = ['GaP']
    for name in dbname:
        loadDB(name)

Cloning database GaP
GaP/DetectorGeo.sql
GaP/PmtBlr.sql
GaP/ChannelGain.sql
GaP/ChannelMapping.sql
GaP/ChannelMask.sql
GaP/PmtNoiseRms.sql
GaP/ChannelPosition.sql
GaP/PMTFEMapping.sql
GaP/PMTFELowFrequencyNoise.sql


### Check values

In [1]:
import gres.database.load_db as db

In [3]:
db.DataPMT('gap', 2965)

,SensorID,ChannelID,PmtID,Active,X,Y,coeff_blr,coeff_c,adc_to_pes,noise_rms,Sigma
0,0,0,PMT,1,0.000,0.000,0,0,8.67,0,3.51
1,1,1,PMT,1,36.253,2.949,0,0,7.60,0,4.14
2,2,2,PMT,1,-15.573,-32.871,0,0,5.54,0,-3.15
3,3,3,PMT,1,-20.680,29.922,0,0,10.47,0,4.43
4,4,4,PMT,1,20.680,-29.922,0,0,10.34,0,5.54
5,5,5,PMT,1,-36.253,-2.949,0,0,7.64,0,4.52
6,6,6,PMT,1,15.573,32.871,0,0,13.73,0,6.58


In [2]:
db.DataPMT('gap', 2965)

,SensorID,ChannelID,PmtID,Active,X,Y,coeff_blr,coeff_c,adc_to_pes,noise_rms,Sigma
0,0,0,PMT,1,0.000,0.000,0,0,8.67,0,3.51
1,1,1,PMT,1,36.253,2.949,0,0,7.60,0,4.14
2,2,2,PMT,1,20.680,-29.922,0,0,5.54,0,-3.15
3,3,3,PMT,1,-15.573,-32.871,0,0,10.47,0,4.43
4,4,4,PMT,1,-36.253,-2.949,0,0,10.34,0,5.54
5,5,5,PMT,1,-20.680,29.922,0,0,7.64,0,4.52
6,6,6,PMT,1,15.573,32.871,0,0,13.73,0,6.58


In [9]:
db.DataPMT('gap', 2452)

,SensorID,ChannelID,PmtID,Active,X,Y,coeff_blr,coeff_c,adc_to_pes,noise_rms,Sigma
0,0,0,PMT,1,0.000,0.000,0,0,11.12,0,8.18
1,1,1,PMT,1,36.253,2.949,0,0,7.30,0,-6.39
2,2,2,PMT,1,-15.573,-32.871,0,0,14.71,0,6.94
3,3,3,PMT,1,-20.680,29.922,0,0,8.52,0,6.03
4,4,4,PMT,1,20.680,-29.922,0,0,13.11,0,7.69
5,5,5,PMT,1,-36.253,-2.949,0,0,16.09,0,7.20
6,6,6,PMT,1,15.573,32.871,0,0,12.59,0,7.92


In [5]:
db.DataPMT('gap', 1073)


,SensorID,ChannelID,PmtID,Active,X,Y,coeff_blr,coeff_c,adc_to_pes,noise_rms,Sigma
0,0,0,PMT,1,0.000,0.000,0,0,11.12,0,8.18
1,1,1,PMT,1,36.253,2.949,0,0,7.30,0,-6.39
2,2,2,PMT,1,-15.573,-32.871,0,0,14.71,0,6.94
3,3,3,PMT,1,-20.680,29.922,0,0,8.52,0,6.03
4,4,4,PMT,1,20.680,-29.922,0,0,13.11,0,7.69
5,5,5,PMT,1,-36.253,-2.949,0,0,16.09,0,7.20
6,6,6,PMT,1,15.573,32.871,0,0,12.59,0,7.92
